# Making SIMSOPT GPU native: SciPy trajectory comparison

Select **Runtime > Change runtime type > GPU**, then run all cells. This compares 25 L-BFGS-B iterations of the complete stress-scale engineering objective through the existing SIMSOPT CPU graph and the compiled GPU bridge. It checks identical variable ordering, accepted iterates, final engineering metrics, physics-evaluation count, and end-to-end optimization speed.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-trajectory-profile")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
result_file = artifact_root / "stress-scipy-trajectory.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/compare_scipy_trajectories.py", "--problem", "stress", "--maxiter", "25", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_file)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_file.read_text())
print(json.dumps({"all_gates_passed": result["all_gates_passed"], "comparison": result["comparison"], "gates": result["gates"]}, indent=2))
assert result["objective"]["scope"] == "gpu_native_full_engineering"
assert result["objective"]["deferred_terms"] == []
assert result["all_gates_passed"], result["gates"]

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-trajectory-profile", "zip", artifact_root)
files.download(archive)